# BrailleVision — YOLOv8 Training on Google Colab
**Run all cells top to bottom. Takes ~60-90 min on a free T4 GPU.**

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')
!nvidia-smi | head -10

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────
!pip install -q ultralytics opencv-python-headless numpy pyyaml
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Clone repo ───────────────────────────────────────────
import os
if not os.path.exists('/content/BrailleVision'):
    !git clone https://github.com/VitthalGund/BrailleVision.git /content/BrailleVision
else:
    print('Repo already cloned. Pulling latest...')
    !cd /content/BrailleVision && git pull

%cd /content/BrailleVision
!ls

In [ ]:
# ── Cell 4: Download real Braille datasets ─────────────────────────
!python training/scripts/download_datasets.py --skip-synthetic
print('\nReal datasets done.')

In [ ]:
# ── Cell 5: Generate 800 synthetic Braille images ──────────────────
!python training/scripts/generate_synthetic.py --count 800
print('\nSynthetic generation done.')

In [ ]:
# ── Cell 6: Merge & split into train/val/test ──────────────────────
!python training/scripts/merge_and_split.py
!python training/scripts/verify_dataset.py
print('\nDataset ready.')
!cat dataset/data.yaml

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────
# ~60-90 min on T4 GPU
!python training/scripts/train.py \
    --epochs 100 \
    --batch 16 \
    --device 0 \
    --name braille_dot_v1

In [ ]:
# ── Cell 8: Evaluate ──────────────────────────────────────────────
!python training/scripts/evaluate.py --skip-e2e
!cat training/runs/eval_report.md

In [ ]:
# ── Cell 9: Download weights ──────────────────────────────────────
from google.colab import files
import os

if os.path.exists('model/best.pt'):
    files.download('model/best.pt')
    print('Downloaded best.pt')
else:
    print('ERROR: best.pt not found. Check training output above.')

if os.path.exists('model/model.onnx'):
    files.download('model/model.onnx')
    print('Downloaded model.onnx')

print('\nPlace both files in the model/ folder of your local project.')